In [1]:
import pandas as pd
import datetime as dt
from datetime import datetime
from dataretrieval import nwis
from dataretrieval import waterdata
from Python_Files import data
from shapely.geometry import Point, LineString
from shapely.ops import substring
import numpy as np
import matplotlib.pyplot as plt
import pynhd as nhd
from pynhd import NLDI
import geopandas as gpd

In [2]:
upstream_id = "10155200"
downstream_id = "10155500"
start_date = '2016-01-01'
end_date = '2025-12-31'
sites = [upstream_id,downstream_id]
upstream_coord = Point(-111.433242586696,40.5543980519593) #from USGS page
downstream_coord = Point(-111.463519813752,40.4841221082783)#from USGS page



## Channel Data and Geometry 

In [3]:
discharge = data.get_discharge(upstream_id,downstream_id,start_date,end_date)
discharge.head()

,Upstream Mean Discharge (cfs),Downstream Mean Discharge (cfs)
Date,,
2016-01-01,137.0,146.0
2016-01-02,134.0,147.0
2016-01-03,136.0,146.0
2016-01-04,135.0,144.0
2016-01-05,134.0,145.0


In [4]:
width = data.get_width(upstream_id,downstream_id,start_date,end_date)
width.head()

,Downstream Time,Downstream Width (ft),Upstream Time,Upstream Width (ft)
Date,,,,
2016-01-12,NaN,NaN,08:26:00,65.0
2016-01-14,08:15:00,57.0,NaN,NaN
2016-02-26,09:47:00,58.0,07:20:00,65.0
2016-04-06,07:59:00,58.5,NaN,NaN
2016-04-07,NaN,NaN,09:34:00,65.0


In [5]:
g2d = data.get_depth_2_datum(upstream_id,downstream_id,start_date,end_date)
g2d.head()

,Upstream Time,Upstream to Datum (ft),Downstream Time,Downstream to Datum (ft)
Date,,,,
2016-01-12,08:26:00,0.8150,NaN,NaN
2016-01-14,NaN,NaN,08:15:00,3.5776
2016-02-26,07:20:00,0.8250,09:47:00,3.6768
2016-04-06,NaN,NaN,07:59:00,3.5412
2016-04-07,09:34:00,0.7936,NaN,NaN


In [6]:
start = datetime.strptime(start_date, "%Y-%m-%d")
end = datetime.strptime(end_date, "%Y-%m-%d")
years = np.array(range(start.year, end.year + 1)).reshape(-1, 2)
surface = data.get_surface(upstream_id,downstream_id,years)

In [ ]:
depth = g2d
depth['Upstream to Datum (ft)'] = depth['Upstream to Datum (ft)']+surface['Upstream Gauge Depth (ft)']
depth['Downstream to Datum (ft)'] = depth['Downstream to Datum (ft)']+surface['Downstream Gauge Depth (ft)']
depth = depth.rename(columns={'Upstream to Datum (ft)':'Upstream Depth (ft)','Downstream to Datum (ft)':'Downstream Depth (ft)'})
depth.to_csv('../Data Files/Raw/depth.csv')

## Slope field to estimate using NLDI

In [ ]:
#nldi will work upstream, so using downstream for this code block
nldi = NLDI()

basin = nldi.get_basins(downstream_id)
flowlines = nldi.navigate_byid(
        fsource="nwissite",
        fid=f"USGS-{downstream_id}",
        navigation='upstreamMain',
        source='flowlines',
        distance=200,
    )

vaa = nhd.nhdplus_vaa("input_data/nhdplus_vaa.parquet")
flowlines["comid"] = pd.to_numeric(flowlines.nhdplus_comid)
slope = gpd.GeoDataFrame(
    pd.merge(flowlines, vaa[["comid", "slope"]], left_on="comid", right_on="comid"),
    crs=flowlines.crs,
)
slope[slope.slope < 0] = np.nan

In [ ]:
slopes = slope.head(5)
slopes.to_csv('../Data Files/Raw/slopes.csv')

Using a map website (such as Google earth) and looking for the slope coordinates in the dataframe that crosses each station, it was found that Downstream slope = 0.006312 (index 0) and Upstream slope = 0.009211 (index 4)

Length of the reach was found through USGS streamstats page, found to be about 42661.29 ft. This is an approximation. 